# 16 — Estudo integrador: MovieLens e Book-Crossing

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 16/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/16_estudo_integrador_movielens_livros.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_16_estudo_integrador_movielens_livros.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Comparar dois domínios
- Separar feedback explícito e implícito no Book-Crossing
- Apresentar baseline robusto e protocolo para evolução

## Fundamentação conceitual
MovieLens contém notas explícitas. No Book-Crossing, `Book-Rating = 0` representa interação **implícita** e não deve entrar na média de notas explícitas. Quando existem amostras autorizadas locais, elas são usadas como estudo de Book-Crossing e não representam necessariamente a população. Sem elas, o notebook usa uma fixture sintética explícita, **não** o conjunto Book-Crossing real.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## MovieLens: ranking com suporte

In [2]:
st=ratings.groupby('movieId').rating.agg(['mean','count'])
mu=ratings.rating.mean();st['smooth']=(st['count']*st['mean']+30*mu)/(st['count']+30)
print(movies.set_index('movieId').loc[st.sort_values('smooth',ascending=False).head(5).index,'title'].tolist())

['Shawshank Redemption, The (1994)', 'Godfather, The (1972)', 'Fight Club (1999)', 'Star Wars: Episode IV - A New Hope (1977)', 'Usual Suspects, The (1995)']


**Interpretação.** O baseline regularizado pode ser aplicado a outro domínio com cuidado.

## Book-Crossing: carregar amostra

In [3]:
# EP_DATASETS_09_F1 — dados locais autorizados OU fixture didática sintética.
# A fixture NÃO é Book-Crossing real e jamais deve ser citada como tal.
caminho_notas = DATA / "bx_ratings_amostra.csv"
caminho_livros = DATA / "bx_books_amostra.csv"
if caminho_notas.exists() and caminho_livros.exists():
    origem_livros = "book_crossing_local"
    bx = pd.read_csv(caminho_notas)
    books = pd.read_csv(caminho_livros)
elif caminho_notas.exists() != caminho_livros.exists():
    raise FileNotFoundError(
        "Apenas uma amostra Book-Crossing foi encontrada. "
        "Inclua ambas as amostras autorizadas ou retire ambas para usar a fixture sintética."
    )
else:
    origem_livros = "sintetico"
    books = pd.DataFrame({
        "ISBN": ["SYN-001", "SYN-002", "SYN-003", "SYN-004", "SYN-005"],
        "Book-Title": ["[SINTÉTICO] Livro A", "[SINTÉTICO] Livro B",
                       "[SINTÉTICO] Livro C", "[SINTÉTICO] Livro D",
                       "[SINTÉTICO] Livro E"],
    })
    bx = pd.DataFrame({
        "User-ID": [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6],
        "ISBN": ["SYN-001", "SYN-002", "SYN-001", "SYN-003",
                 "SYN-002", "SYN-004", "SYN-001", "SYN-005",
                 "SYN-003", "SYN-004", "SYN-002", "SYN-005"],
        "Book-Rating": [8, 0, 9, 7, 6, 0, 10, 8, 0, 5, 7, 9],
    })
    print("ATENÇÃO: dados SINTÉTICOS demonstrativos. Nenhuma métrica Book-Crossing real foi calculada.")
for col in ("ISBN", "Book-Rating"):
    if col not in bx.columns:
        raise ValueError(f"Coluna ausente em avaliações de livros: {col}")
for col in ("ISBN", "Book-Title"):
    if col not in books.columns:
        raise ValueError(f"Coluna ausente no catálogo de livros: {col}")
bx["Book-Rating"] = pd.to_numeric(bx["Book-Rating"], errors="raise")
assert bx["Book-Rating"].between(0, 10).all(), "Notas de livros fora do intervalo 0–10."
assert bx["Book-Rating"].gt(0).any(), "Sem avaliações explícitas para construir ranking."
print("Fonte usada:", origem_livros, "| Registros:", len(bx), "| Livros:", len(books))
print("Amostra de interações:", bx.head(3).to_string(index=False))


Fonte usada: book_crossing_local | Registros: 15000 | Livros: 179
Amostra de interações:  User-ID       ISBN  Book-Rating
  276755 0451166892            5
  276788 043935806X            7
  276847 0446364193            0


**Interpretação.** Os dados originais foram distribuídos em CSV com delimitador ponto e vírgula; a amostra foi normalizada.

## Separar eventos implícitos e explícitos

In [4]:
implicit=bx[bx['Book-Rating'].eq(0)].copy()
explicit=bx[bx['Book-Rating'].gt(0)].copy()
print('Feedback implícito (0):',len(implicit),'| Explícito (1–10):',len(explicit))
assert explicit['Book-Rating'].between(1,10).all()

Feedback implícito (0): 8437 | Explícito (1–10): 6563


**Interpretação.** Não misture o zero com notas baixas no cálculo da média.

## Ranking de livros com nota explícita

In [5]:
st=explicit.groupby('ISBN')['Book-Rating'].agg(['mean','count'])
mu=explicit['Book-Rating'].mean();st['score']=(st['mean']*st['count']+10*mu)/(st['count']+10)
results=st.sort_values('score',ascending=False).head(8).reset_index().merge(books,on='ISBN',how='left')
print(results[['Book-Title','count','score']].to_string(index=False))

                                                      Book-Title  count    score
              Harry Potter and the Order of the Phoenix (Book 5)     40 8.968216
Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))     70 8.905135
                                           To Kill a Mockingbird     64 8.856903
                                         A Prayer for Owen Meany     45 8.825651
                Harry Potter and the Chamber of Secrets (Book 2)     44 8.711311
                                               The Da Vinci Code    115 8.683286
                                         The Secret Life of Bees     68 8.671933
                                                  Fahrenheit 451     42 8.642515


**Interpretação.** Quando usados dados Book-Crossing reais, a forma de amostragem limita a representatividade. Na execução sintética, o ranking apenas exemplifica o cálculo, sem inferências sobre usuários/livros reais.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** separação de domínios. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **separação de domínios** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Separar nota zero de nota explícita em Book-Crossing

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [6]:
import pandas as pd
exemplo=pd.DataFrame({'livro':['A','B','C'],'nota':[0,7,10]})
explicitas=exemplo.loc[exemplo.nota>0]
print('Interações totais:',len(exemplo),'notas explícitas:',len(explicitas))
assert len(explicitas)==2

Interações totais: 3 notas explícitas: 2


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **separação de domínios** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Separação de domínios e feedback

MovieLens representa avaliação explícita em escala 0,5–5; Book-Crossing possui escala 1–10 para avaliações explícitas e código **zero para interação implícita** na convenção do conjunto. Quando usadas amostras locais reais, a seleção pode enviesar a distribuição; a fixture sintética não permite estimar qualquer métrica do domínio Book-Crossing. Não compare diretamente as escalas entre domínios.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [7]:
# Conferência transparente dos sinais no Book-Crossing
qtd_impl = int(bx['Book-Rating'].eq(0).sum())
qtd_expl = int(bx['Book-Rating'].between(1,10).sum())
print({'implícitas_zero':qtd_impl,'explícitas_1_a_10':qtd_expl,'total':len(bx)})
assert qtd_impl+qtd_expl == len(bx)

{'implícitas_zero': 8437, 'explícitas_1_a_10': 6563, 'total': 15000}


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
A execução com amostras Book-Crossing locais pode conter vieses de seleção; a execução com fixture sintética ilustra apenas a metodologia, sem medir desempenho em dados reais.

## Exercícios
Explique por que não devemos comparar diretamente a nota 4 do MovieLens à nota 4 do Book-Crossing.

## Soluções comentadas
As escalas diferem (MovieLens 0.5–5; livros 1–10) e populações/viés de seleção não coincidem.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados em `outputs/`. Na ausência de amostras locais autorizadas, os nomes recebem `_sintetico` e o campo `origem` distingue a fixture demonstrativa de observações reais.


In [8]:
from pathlib import Path
out = Path("outputs")
out.mkdir(parents=True, exist_ok=True)
sufixo = "_sintetico" if origem_livros == "sintetico" else ""
ranking = results[["ISBN", "Book-Title", "count", "score"]].rename(columns={
    "count": "n_notas_explicitas", "score": "media_suavizada_explicita"
}).assign(origem=origem_livros)
ranking.to_csv(out / f"16_ranking_livros_amostra{sufixo}.csv", index=False)
contagem = pd.DataFrame([{
    "feedback_implicito_zero": len(implicit),
    "feedback_explicito_1_a_10": len(explicit),
    "total_interacoes": len(bx),
    "origem": origem_livros,
}])
contagem.to_csv(out / f"16_resumo_feedback_livros{sufixo}.csv", index=False)
print("Outputs do capítulo 16:", origem_livros, "| arquivos em", out.resolve())


Outputs do capítulo 16: book_crossing_local | arquivos em /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
